# 02 — Baselines

This notebook establishes baseline forecasts using the shared walk-forward
backtesting harness (`src/evaluation.py`). Every later model (LightGBM, LSTM,
TFT) gets compared against these numbers — a model that can't beat a seasonal
naive forecast isn't earning its complexity.

**Why start with such simple models?** Because "our LSTM gets 4% MAPE" means
nothing on its own. "Our LSTM gets 4% MAPE, beating seasonal naive's 7%"
is an actual finding. Baselines are what make later results interpretable.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

from src.evaluation import run_backtest, summarize

plt.rcParams["figure.figsize"] = (14, 4)

df = pd.read_csv("../data/processed/load_weather_hourly.csv", index_col=0, parse_dates=True)
df.index.name = "timestamp"
df = df.sort_index()
print(df.shape)
df.head()

## 1. Naive forecast

Predicts: "load in the next hour = load right now." The simplest possible
baseline. If a sophisticated model can't beat this, something is wrong.


In [ ]:
def naive_forecast(train_df, test_df):
    last_value = train_df["load_MW"].iloc[-1]
    return np.full(len(test_df), last_value)

naive_result = run_backtest(df, naive_forecast)
naive_summary = summarize(naive_result, "Naive")
naive_summary

## 2. Seasonal naive forecast

Predicts: "load at this hour = load at the *same hour, same day-of-week,*
168 hours (one week) ago." This is a much stronger baseline than plain naive
for data with strong weekly seasonality — expect it to beat naive by a wide
margin, and expect it to be genuinely hard for early ML models to beat.


In [ ]:
def seasonal_naive_forecast(train_df, test_df, season_lag=168):
    full = pd.concat([train_df, test_df])
    preds = []
    for ts in test_df.index:
        lookback_ts = ts - pd.Timedelta(hours=season_lag)
        if lookback_ts in full.index:
            preds.append(full.loc[lookback_ts, "load_MW"])
        else:
            preds.append(train_df["load_MW"].iloc[-1])  # fallback
    return np.array(preds)

seasonal_naive_result = run_backtest(df, seasonal_naive_forecast)
seasonal_naive_summary = summarize(seasonal_naive_result, "Seasonal Naive (168h)")
seasonal_naive_summary

## 3. SARIMA

Classical statistical model with explicit trend (p,d,q) and seasonal
(P,D,Q,s) orders. `s=24` sets the seasonal period to daily.

**Note on runtime:** SARIMA is fit fresh in every fold, and it's noticeably
slower than the naive baselines — this cell may take a while depending on
how many folds `run_backtest`'s default `step` produces. If it's too slow,
increase `step` (fewer folds) or reduce `horizon`, and say so explicitly in
your report as a documented trade-off, not a hidden shortcut.


In [ ]:
def sarima_forecast(train_df, test_df, order=(2, 0, 2), seasonal_order=(1, 1, 1, 24)):
    # Cap training history for speed — SARIMA scales poorly with series length.
    # Using the most recent ~60 days is a reasonable, statable trade-off.
    train_tail = train_df["load_MW"].iloc[-24*60:]
    model = SARIMAX(train_tail, order=order, seasonal_order=seasonal_order,
                     enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    forecast = fit.forecast(steps=len(test_df))
    return forecast.values

# Use a larger step to keep runtime reasonable given SARIMA's cost per fold
sarima_result = run_backtest(df, sarima_forecast, step=24*30)  # roughly monthly folds
sarima_summary = summarize(sarima_result, "SARIMA")
sarima_summary

## 4. Compare baselines

This table is the reference point for every model built in later notebooks.


In [ ]:
comparison = pd.DataFrame([naive_summary, seasonal_naive_summary, sarima_summary])
comparison = comparison.sort_values("mean_mape")
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison["model"], comparison["mean_mape"])
ax.set_ylabel("Mean MAPE (%)")
ax.set_title("Baseline Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 5. Inspect a sample fold visually

Numbers alone don't show *how* a model is wrong (systematically high? bad at
peaks? lagging behind sudden changes?). Always look at at least one fold's
actual predicted-vs-true plot before trusting a metric.


In [ ]:
sample_fold = seasonal_naive_result.predictions[seasonal_naive_result.predictions["fold"] == 0]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample_fold["timestamp"], sample_fold["y_true"], label="Actual")
ax.plot(sample_fold["timestamp"], sample_fold["y_pred"], label="Seasonal Naive Prediction")
ax.legend()
ax.set_title("Seasonal Naive — Sample Fold")
plt.tight_layout()
plt.show()

## Summary

_(Fill in after running: which baseline won, by how much, and does the
sample-fold plot reveal any systematic pattern in the errors — e.g. does the
model consistently miss the morning peak, or lag behind fast temperature
swings? This directly motivates what the ML model in `03_ml_models.ipynb`
needs to fix.)_

## Next steps

`03_ml_models.ipynb`: build the LightGBM model using `src/features.py`,
backtest with the same harness, and compare against this table.
